In [2]:
#Import Libraries
import pandas as pd
import numpy as np
import os
import sqlite3

from sqlalchemy import create_engine

In [3]:
#Create Database Connection
engine = create_engine('sqlite:///inventory.db')

print("Database created successfully")

Database created successfully


In [4]:
#Load All CSV Files into SQLite
folder_path = r"C:\Users\vaide\Downloads\data\data"

for file in os.listdir(folder_path):

    if file.endswith('.csv'):

        file_path = os.path.join(folder_path, file)

        df = pd.read_csv(file_path)

        table_name = file.replace('.csv', '')

        df.to_sql(
            table_name,
            con=engine,
            if_exists='replace',
            index=False
        )

        print(f"{table_name} loaded successfully")

begin_inventory loaded successfully
end_inventory loaded successfully
purchases loaded successfully
purchase_prices loaded successfully


MemoryError: 

In [ ]:
#Verify Tables Were Created
with engine.connect() as conn:

    tables = pd.read_sql("""
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """, conn)

tables


In [ ]:
#Inspect First Few Rows

with engine.connect() as conn:

    sales = pd.read_sql(
        "SELECT * FROM sales LIMIT 5",
        conn
    )

sales

In [ ]:
#Check Shapes of All Tables
tables = [
    'begin_inventory',
    'end_inventory',
    'purchases',
    'purchase_prices'
]

for table in tables:

    with engine.connect() as conn:

        query = f"""
        SELECT COUNT(*) as total_rows
        FROM {table}
        """

        df = pd.read_sql(query, conn)

        print(table)
        print(df)
        print("-"*30)

In [ ]:
#Vendor Purchase Summary
query = """

SELECT

VendorNumber,
VendorName,

SUM(PurchasePrice * Quantity) AS TotalPurchaseAmount,
SUM(Quantity) AS TotalPurchaseQty

FROM purchases

GROUP BY VendorNumber, VendorName

"""

with engine.connect() as conn:

    purchase_summary = pd.read_sql(query, conn)

purchase_summary.head()

In [ ]:
#Sales Summary
query = """

SELECT

VendorNo,
SUM(SalesDollars) AS TotalSales,
SUM(SalesQuantity) AS TotalSalesQty

FROM sales

GROUP BY VendorNo

"""

with engine.connect() as conn:

    sales_summary = pd.read_sql(query, conn)

sales_summary.head()


In [ ]:
#Vendor Invoice Summary

query = """

SELECT

VendorNumber,
SUM(Freight) AS TotalFreight

FROM vendor_invoice

GROUP BY VendorNumber

"""

with engine.connect() as conn:

    freight_summary = pd.read_sql(query, conn)

freight_summary.head()

In [ ]:
import os

folder_path = r"C:\Users\vaide\Downloads\data\data"

for file in os.listdir(folder_path):
    print(file)

In [ ]:
import pandas as pd
import os

folder_path = r"C:\Users\vaide\Downloads\data\data"

for file in os.listdir(folder_path):

    if file.endswith('.csv'):

        try:
            print(f"\nLoading {file}")

            file_path = os.path.join(folder_path, file)

            df = pd.read_csv(file_path)

            print(df.shape)

        except Exception as e:
            print(f"Error in {file}: {e}")

In [ ]:
import os

file_path = r"C:\Users\vaide\Downloads\data\data\purchases.csv"

size_mb = os.path.getsize(file_path) / (1024 * 1024)

print(f"File size: {size_mb:.2f} MB")

In [16]:
print("Hello World")

Hello World


In [18]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine('sqlite:///inventory.db')

In [24]:
with engine.connect() as conn:

    tables = pd.read_sql("""
        SELECT name
        FROM sqlite_master
        WHERE type='table';
    """, conn)

tables

,name
0,begin_inventory
1,end_inventory
2,purchases
3,purchase_prices
4,sales
5,vendor_invoice


In [22]:
import pandas as pd

vendor_invoice = pd.read_csv(
    r"C:\Users\vaide\Downloads\data\data\vendor_invoice.csv"
)

vendor_invoice.to_sql(
    'vendor_invoice',
    con=engine,
    if_exists='replace',
    index=False
)

print("vendor_invoice loaded successfully")

vendor_invoice loaded successfully


In [26]:
# Establish a connection to the SQLite database
# 'engine' was created earlier using SQLAlchemy
with engine.connect() as conn:

    # Read data from the 'purchases' table
    # SELECT * means fetch all columns
    # LIMIT 5 means fetch only the first 5 rows
    # This is done to inspect the table structure without loading the entire table
    purchases = pd.read_sql(
        """
        SELECT *
        FROM purchases
        LIMIT 5
        """,
        conn
    )

# Display the first 5 rows stored in the DataFrame
purchases

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


In [28]:
# This SQL query summarizes purchase information for each vendor

query = """

SELECT

    -- Unique vendor ID
    VendorNumber,

    -- Vendor name
    VendorName,

    -- Total money spent purchasing from this vendor
    SUM(Dollars) AS TotalPurchaseAmount,

    -- Total quantity purchased from this vendor
    SUM(Quantity) AS TotalPurchaseQty

FROM purchases

-- Group records by vendor so that aggregation happens vendor-wise
GROUP BY VendorNumber, VendorName

"""

# Execute the query and store results in a DataFrame
with engine.connect() as conn:

    purchase_summary = pd.read_sql(query, conn)

# Display first 5 rows
purchase_summary.head()

,VendorNumber,VendorName,TotalPurchaseAmount,TotalPurchaseQty
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",5630.88,328
1,54,AAPER ALCOHOL & CHEMICAL CO,105.07,1
2,60,ADAMBA IMPORTS INTL INC,76770.25,4732
3,105,ALTAMAR BRANDS LLC,11706.20,332
4,200,AMERICAN SPIRITS EXCHANGE,1205.16,132


In [30]:
# Open database connection and fetch first 5 rows from sales table

with engine.connect() as conn:

    sales = pd.read_sql(
        """
        SELECT *
        FROM sales
        LIMIT 5
        """,
        conn
    )

# Display first 5 records
sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName


In [32]:
# This SQL query summarizes sales information for each vendor

query = """

SELECT

    -- Unique vendor ID
    VendorNo,

    -- Vendor name
    VendorName,

    -- Total sales revenue generated by the vendor's products
    SUM(SalesDollars) AS TotalSalesAmount,

    -- Total quantity of products sold
    SUM(SalesQuantity) AS TotalSalesQty

FROM sales

-- Aggregate results vendor-wise
GROUP BY VendorNo, VendorName

"""

# Execute query and save result in a DataFrame
with engine.connect() as conn:

    sales_summary = pd.read_sql(query, conn)

# Display first 5 rows
sales_summary.head()

,VendorNo,VendorName,TotalSalesAmount,TotalSalesQty


In [34]:
# This query calculates total freight cost paid to each vendor

query = """

SELECT

    -- Vendor ID
    VendorNumber,

    -- Total freight charges paid to the vendor
    SUM(Freight) AS TotalFreight

FROM vendor_invoice

-- Aggregate freight vendor-wise
GROUP BY VendorNumber

"""

# Execute query and store results in DataFrame
with engine.connect() as conn:

    freight_summary = pd.read_sql(query, conn)

# Display first few rows
freight_summary.head()

,VendorNumber,TotalFreight
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19


In [36]:
# Calculate total freight cost paid to each vendor

query = """

SELECT

    VendorNumber,

    SUM(Freight) AS TotalFreight

FROM vendor_invoice

GROUP BY VendorNumber

"""

with engine.connect() as conn:

    freight_summary = pd.read_sql(query, conn)

freight_summary.head()

,VendorNumber,TotalFreight
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19


In [38]:
# Merge purchase and sales summaries

vendor_summary = purchase_summary.merge(
    sales_summary,
    left_on='VendorNumber',
    right_on='VendorNo',
    how='left'
)

vendor_summary.head()

,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",5630.88,328,NaN,NaN,NaN,NaN
1,54,AAPER ALCOHOL & CHEMICAL CO,105.07,1,NaN,NaN,NaN,NaN
2,60,ADAMBA IMPORTS INTL INC,76770.25,4732,NaN,NaN,NaN,NaN
3,105,ALTAMAR BRANDS LLC,11706.20,332,NaN,NaN,NaN,NaN
4,200,AMERICAN SPIRITS EXCHANGE,1205.16,132,NaN,NaN,NaN,NaN


In [40]:
# Add freight information

vendor_summary = vendor_summary.merge(
    freight_summary,
    on='VendorNumber',
    how='left'
)

vendor_summary.head()

,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty,TotalFreight
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",5630.88,328,NaN,NaN,NaN,NaN,27.08
1,54,AAPER ALCOHOL & CHEMICAL CO,105.07,1,NaN,NaN,NaN,NaN,0.48
2,60,ADAMBA IMPORTS INTL INC,76770.25,4732,NaN,NaN,NaN,NaN,367.52
3,105,ALTAMAR BRANDS LLC,11706.20,332,NaN,NaN,NaN,NaN,62.39
4,200,AMERICAN SPIRITS EXCHANGE,1205.16,132,NaN,NaN,NaN,NaN,6.19


In [42]:
# Replace missing values with zero

vendor_summary.fillna(0, inplace=True)

vendor_summary.head()

C:\Users\vaide\AppData\Local\Temp\ipykernel_48984\815609664.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vendor_summary.fillna(0, inplace=True)


,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty,TotalFreight
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",5630.88,328,0,0,0,0,27.08
1,54,AAPER ALCOHOL & CHEMICAL CO,105.07,1,0,0,0,0,0.48
2,60,ADAMBA IMPORTS INTL INC,76770.25,4732,0,0,0,0,367.52
3,105,ALTAMAR BRANDS LLC,11706.20,332,0,0,0,0,62.39
4,200,AMERICAN SPIRITS EXCHANGE,1205.16,132,0,0,0,0,6.19


In [44]:
# Gross Profit = Sales Revenue - Purchase Cost

vendor_summary['GrossProfit'] = (
    vendor_summary['TotalSalesAmount']
    - vendor_summary['TotalPurchaseAmount']
)

vendor_summary.head()

,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty,TotalFreight,GrossProfit
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",5630.88,328,0,0,0,0,27.08,-5630.88
1,54,AAPER ALCOHOL & CHEMICAL CO,105.07,1,0,0,0,0,0.48,-105.07
2,60,ADAMBA IMPORTS INTL INC,76770.25,4732,0,0,0,0,367.52,-76770.25
3,105,ALTAMAR BRANDS LLC,11706.20,332,0,0,0,0,62.39,-11706.20
4,200,AMERICAN SPIRITS EXCHANGE,1205.16,132,0,0,0,0,6.19,-1205.16


In [46]:
# Calculate profit margin percentage

vendor_summary['ProfitMargin'] = (
    vendor_summary['GrossProfit']
    / vendor_summary['TotalSalesAmount']
) * 100

vendor_summary.head()

,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty,TotalFreight,GrossProfit,ProfitMargin
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",5630.88,328,0,0,0,0,27.08,-5630.88,-inf
1,54,AAPER ALCOHOL & CHEMICAL CO,105.07,1,0,0,0,0,0.48,-105.07,-inf
2,60,ADAMBA IMPORTS INTL INC,76770.25,4732,0,0,0,0,367.52,-76770.25,-inf
3,105,ALTAMAR BRANDS LLC,11706.20,332,0,0,0,0,62.39,-11706.20,-inf
4,200,AMERICAN SPIRITS EXCHANGE,1205.16,132,0,0,0,0,6.19,-1205.16,-inf


In [48]:
# View first 10 rows

vendor_summary.head(10)

,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty,TotalFreight,GrossProfit,ProfitMargin
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",5630.88,328,0,0,0,0,27.08,-5630.88,-inf
1,54,AAPER ALCOHOL & CHEMICAL CO,105.07,1,0,0,0,0,0.48,-105.07,-inf
2,60,ADAMBA IMPORTS INTL INC,76770.25,4732,0,0,0,0,367.52,-76770.25,-inf
3,105,ALTAMAR BRANDS LLC,11706.20,332,0,0,0,0,62.39,-11706.20,-inf
4,200,AMERICAN SPIRITS EXCHANGE,1205.16,132,0,0,0,0,6.19,-1205.16,-inf
5,287,APPOLO VINEYARDS LLC,2399.70,230,0,0,0,0,12.28,-2399.70,-inf
6,388,ATLANTIC IMPORTING COMPANY,41116.32,1847,0,0,0,0,211.74,-41116.32,-inf
7,480,BACARDI USA INC,17624378.72,1427075,0,0,0,0,89286.27,-17624378.72,-inf
8,516,BANFI PRODUCTS CORP,1628866.68,228103,0,0,0,0,8510.41,-1628866.68,-inf
9,653,STATE WINE & SPIRITS,1529682.04,154092,0,0,0,0,8014.98,-1529682.04,-inf


In [50]:
# Check columns, data types and null values

vendor_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   VendorNumber         129 non-null    int64  
 1   VendorName_x         129 non-null    object 
 2   TotalPurchaseAmount  129 non-null    float64
 3   TotalPurchaseQty     129 non-null    int64  
 4   VendorNo             129 non-null    int64  
 5   VendorName_y         129 non-null    int64  
 6   TotalSalesAmount     129 non-null    int64  
 7   TotalSalesQty        129 non-null    int64  
 8   TotalFreight         129 non-null    float64
 9   GrossProfit          129 non-null    float64
 10  ProfitMargin         129 non-null    float64
dtypes: float64(4), int64(6), object(1)
memory usage: 11.2+ KB


In [52]:
#Top 10 Vendors by Sales

vendor_summary.sort_values(
    by='TotalSalesAmount',
    ascending=False
).head(10)

,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty,TotalFreight,GrossProfit,ProfitMargin
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",5630.88,328,0,0,0,0,27.08,-5630.88,-inf
65,7154,PREMIER DISTRIBUTORS,84530.87,5916,0,0,0,0,432.37,-84530.87,-inf
95,10754,PERFECTA WINES,5665501.53,553116,0,0,0,0,28720.52,-5665501.53,-inf
94,10050,Russian Standard Vodka,138907.27,12798,0,0,0,0,732.60,-138907.27,-inf
93,10000,MAJESTIC FINE WINES,3506668.63,446119,0,0,0,0,17587.59,-3506668.63,-inf
92,9819,TREASURY WINE ESTATES,2978686.40,497770,0,0,0,0,14836.57,-2978686.40,-inf
91,9815,WINE GROUP INC,5258636.79,888385,0,0,0,0,27100.41,-5258636.79,-inf
90,9751,VINEDREA WINES LLC,4657.60,205,0,0,0,0,24.53,-4657.60,-inf
89,9744,FREDERICK WILDMAN & SONS,759449.24,70932,0,0,0,0,3999.93,-759449.24,-inf
88,9625,WESTERN SPIRITS BEVERAGE CO,361249.21,56860,0,0,0,0,1933.19,-361249.21,-inf


In [54]:
#Top 10 Vendors by Gross Profit

vendor_summary.sort_values(
    by='GrossProfit',
    ascending=False
).head(10)

,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty,TotalFreight,GrossProfit,ProfitMargin
128,201359,FLAVOR ESSENCE INC,17.00,1,0,0,0,0,0.09,-17.00,-inf
17,1439,CAPSTONE INTERNATIONAL,54.64,4,0,0,0,0,0.27,-54.64,-inf
110,90026,SILVER MOUNTAIN CIDERS,77.18,17,0,0,0,0,0.36,-77.18,-inf
1,54,AAPER ALCOHOL & CHEMICAL CO,105.07,1,0,0,0,0,0.48,-105.07,-inf
113,90033,FANTASY FINE WINES CORP,128.64,24,0,0,0,0,0.61,-128.64,-inf
50,4901,LAUREATE IMPORTS CO,140.94,29,0,0,0,0,0.72,-140.94,-inf
82,9099,TRUETT HURST,236.64,24,0,0,0,0,1.25,-236.64,-inf
14,1265,BLACK ROCK SPIRITS LLC,1152.10,82,0,0,0,0,5.79,-1152.10,-inf
4,200,AMERICAN SPIRITS EXCHANGE,1205.16,132,0,0,0,0,6.19,-1205.16,-inf
55,5612,MILTONS DISTRIBUTING CO,1824.18,267,0,0,0,0,9.46,-1824.18,-inf


In [56]:
#Vendors with Highest Freight Cost

vendor_summary.sort_values(
    by='TotalFreight',
    ascending=False
).head(10)

,VendorNumber,VendorName_x,TotalPurchaseAmount,TotalPurchaseQty,VendorNo,VendorName_y,TotalSalesAmount,TotalSalesQty,TotalFreight,GrossProfit,ProfitMargin
42,3960,DIAGEO NORTH AMERICA INC,50959796.85,5459788,0,0,0,0,257032.07,-50959796.85,-inf
44,4425,MARTIGNETTI COMPANIES,27821473.91,2637275,0,0,0,0,144929.24,-27821473.91,-inf
45,4425,MARTIGNETTI COMPANIES,40216.11,3136,0,0,0,0,144929.24,-40216.11,-inf
98,12546,JIM BEAM BRANDS COMPANY,24203151.05,2737165,0,0,0,0,123880.97,-24203151.05,-inf
102,17035,PERNOD RICARD USA,24124091.56,1647558,0,0,0,0,123780.22,-24124091.56,-inf
7,480,BACARDI USA INC,17624378.72,1427075,0,0,0,0,89286.27,-17624378.72,-inf
16,1392,CONSTELLATION BRANDS INC,15573917.90,2325892,0,0,0,0,79528.99,-15573917.90,-inf
12,1128,BROWN-FORMAN CORP,13529433.08,1006122,0,0,0,0,68601.68,-13529433.08,-inf
83,9165,ULTRA BEVERAGE COMPANY LLP,13210613.93,1077527,0,0,0,0,68054.70,-13210613.93,-inf
36,3252,E & J GALLO WINERY,12289608.09,1858260,0,0,0,0,61966.91,-12289608.09,-inf


In [58]:

# Save final dataset for Power BI

vendor_summary.to_csv(
    'vendor_sales_summary.csv',
    index=False
)

print("vendor_sales_summary.csv saved successfully")

vendor_sales_summary.csv saved successfully


In [60]:
# Verify Saved File
import os

print(os.getcwd())

C:\Users\vaide
